In [3]:
!pip install openai-whisper sentence-transformers spacy soundfile pandas numpy --quiet
!python -m spacy download en_core_web_sm --quiet
!apt-get install ffmpeg -y -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 14.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 39.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

dataset_path = Path("/content/drive/MyDrive/diplom/dataset")

print([p.name for p in dataset_path.iterdir()])

Mounted at /content/drive
['video1', 'video2', 'video3', 'video4', 'video5', 'video6', 'video7', 'video8', 'video9', 'video10', 'video11', 'video12', 'video13', 'video14', 'video15']


In [2]:
import subprocess

def extract_audio(video_path, output_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", str(video_path),
        "-vn",
        "-acodec", "pcm_s16le",
        "-ac", "1",
        "-ar", "16000",
        str(output_path),
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


video_folders = sorted([p for p in dataset_path.iterdir() if p.is_dir()])

data = []

for folder in video_folders:
    video_path = folder / "video.mp4"
    audio_path = folder / "audio.wav"
    ref_path = folder / "en.srt"
    extract_audio(video_path, audio_path)

    data.append({
        "folder": folder.name,
        "video": video_path,
        "audio": audio_path,
        "ref": ref_path
    })

print("Готово:", len(data), "видео")

Готово: 15 видео


ASR

In [4]:
import whisper

class WhisperASR:
    def __init__(self, model_size: str = "small"):
        self.device = "cpu"
        print(f"Инициализация Whisper ({model_size})... Устройство: {self.device}")
        self.model = whisper.load_model(model_size, device=self.device)

    def transcribe(self, audio_path: str):
        print("Запуск распознавания...")

        result = self.model.transcribe(
            audio_path,
            language="en",
            task="transcribe",
            fp16=False,
        )

        raw_text = result["text"].strip()
        segments = result["segments"]

        return raw_text, segments


In [5]:
import time

asr = WhisperASR(model_size="small")

asr_results = []

for item in data:
    print(f"\nОбрабатывается: {item['folder']}")

    start_time = time.time()

    raw_text, segments = asr.transcribe(str(item["audio"]))

    elapsed_time = time.time() - start_time

    asr_results.append({
        "folder": item["folder"],
        "video": item["video"],
        "audio": item["audio"],
        "ref": item["ref"],
        "raw_text": raw_text,
        "segments": segments,
        "asr_time_sec": round(elapsed_time, 2),
    })

    print("Сегментов Whisper:", len(segments))
    print("Время распознавания:", round(elapsed_time, 2), "сек")
    print("Первые 200 символов:", raw_text[:200])

Инициализация Whisper (small)... Устройство: cpu

Обрабатывается: video1
Запуск распознавания...
Сегментов Whisper: 198
Время распознавания: 602.77 сек
Первые 200 символов: I have a confession to make. But first, I want you to make a little confession to me. In the past year, I want you to just raise your hand if you've experienced relatively little stress. Anyone? How a

Обрабатывается: video10
Запуск распознавания...
Сегментов Whisper: 26
Время распознавания: 77.0 сек
Первые 200 символов: When you're a kid, I think the universe revolves around you. You think that you'll always be protected and cared for. Then, one day, you realize that's not true. Because when you're alone as a kid, th

Обрабатывается: video11
Запуск распознавания...
Сегментов Whisper: 37
Время распознавания: 229.59 сек
Первые 200 символов: Prince Meliega and his band of warriors crept towards the den of the terrifying beast they sought to slay. But unbeknownst to the prince and his followers, a dark secret loomed ov

Сохранение результатов ASR

In [6]:
import pickle

whisper_results = asr_results

save_path = "/content/drive/MyDrive/diplom/whisper_results_segments.pkl"

with open(save_path, "wb") as f:
    pickle.dump(whisper_results, f)

print("Whisper results сохранены:", save_path)

Whisper results сохранены: /content/drive/MyDrive/diplom/whisper_results_segments.pkl


Генерация базовых субтитров

In [14]:
import re
import textwrap
import time
from typing import List


MAX_LINE_LEN = 39
MAX_LINES = 2
MAX_CHARS_BLOCK = MAX_LINE_LEN * MAX_LINES
MAX_READING_SPEED = 17
MIN_BLOCK_DUR = 1.0
MIN_GAP_BETWEEN_SUBS = 1.0


def clean_text_for_subs(text: str) -> str:
    text = re.sub(r"\b(\w+)([\s,]+\1\b)+", r"\1", text, flags=re.IGNORECASE)

    fillers = r"\b(er|eh|hmm|mm-hmm|uh-huh|uh-uh|oh|ah|uh|huh|erm|um)\b[,\s]*"
    text = re.sub(fillers, " ", text, flags=re.IGNORECASE)

    text = re.sub(r"\s{2,}", " ", text).strip()
    return text


def split_text_into_lines(text: str, max_line_len: int = MAX_LINE_LEN) -> List[str]:
    return textwrap.wrap(text, width=max_line_len)


def is_valid_block_text(
    text: str,
    max_chars_block: int = MAX_CHARS_BLOCK,
    max_line_len: int = MAX_LINE_LEN,
    max_lines: int = MAX_LINES,
) -> bool:
    lines = split_text_into_lines(text, max_line_len=max_line_len)

    return (
        len(text) <= max_chars_block
        and len(lines) <= max_lines
        and all(len(line) <= max_line_len for line in lines)
    )


def split_segment_text_into_blocks(
    text: str,
    max_chars_block: int = MAX_CHARS_BLOCK,
    max_line_len: int = MAX_LINE_LEN,
    max_lines: int = MAX_LINES,
) -> List[str]:
    text = clean_text_for_subs(text)

    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if s.strip()]

    blocks: List[str] = []

    for sent in sentences:
        content_only = re.sub(r"[\W_]+", "", sent, flags=re.UNICODE)
        if not content_only:
            continue

        words = sent.split()
        current_words: List[str] = []

        for w in words:
            test_block = " ".join(current_words + [w])

            if not is_valid_block_text(
                test_block,
                max_chars_block=max_chars_block,
                max_line_len=max_line_len,
                max_lines=max_lines,
            ):
                if current_words:
                    blocks.append(" ".join(current_words))
                    current_words = [w]
                else:
                    current_words = [w]
            else:
                current_words.append(w)

        if current_words:
            blocks.append(" ".join(current_words))

    return blocks


def adjust_timings_for_reading_speed(
    blocks,
    max_reading_speed: float = MAX_READING_SPEED,
    min_gap: float = MIN_GAP_BETWEEN_SUBS,
):


    if not blocks:
        return blocks

    adjusted = [b.copy() for b in blocks]

    for i, block in enumerate(adjusted):
        text_len = len(block["text"])
        current_start = block["start"]
        current_end = block["end"]
        current_duration = current_end - current_start

        if current_duration <= 0:
            continue

        current_speed = text_len / current_duration

        if current_speed <= max_reading_speed:
            continue

        needed_duration = text_len / max_reading_speed
        desired_end = current_start + needed_duration

        if i < len(adjusted) - 1:
            next_start = adjusted[i + 1]["start"]
            max_possible_end = next_start - min_gap
        else:
            max_possible_end = desired_end

        if max_possible_end > current_end:
            new_end = min(desired_end, max_possible_end)

            if new_end > current_end:
                block["end"] = new_end
                block["timing_adjusted"] = True
            else:
                block["timing_adjusted"] = False
        else:
            block["timing_adjusted"] = False

    return adjusted


def segments_to_blocks(
    segments,
    max_chars_block: int = MAX_CHARS_BLOCK,
    max_line_len: int = MAX_LINE_LEN,
    max_lines: int = MAX_LINES,
    min_block_dur: float = MIN_BLOCK_DUR,
    adjust_reading_time: bool = True,
):
    subtitle_blocks = []

    for seg in segments:
        seg_start = seg["start"]
        seg_end = seg["end"]
        seg_text = seg["text"].strip()

        if not seg_text:
            continue

        blocks = split_segment_text_into_blocks(
            seg_text,
            max_chars_block=max_chars_block,
            max_line_len=max_line_len,
            max_lines=max_lines,
        )

        if not blocks:
            continue

        seg_duration = seg_end - seg_start
        total_chars = sum(len(b) for b in blocks)

        if total_chars == 0 or seg_duration <= 0:
            continue

        n_blocks = len(blocks)

        base_durs = [(len(b) / total_chars) * seg_duration for b in blocks]

        if seg_duration >= n_blocks * min_block_dur:
            extra_time = seg_duration - n_blocks * min_block_dur
            base_sum = sum(base_durs) or 1.0
            block_durs = [
                min_block_dur + extra_time * (bd / base_sum)
                for bd in base_durs
            ]
        else:
            block_durs = base_durs

        dur_sum = sum(block_durs)

        if dur_sum > 0:
            scale = seg_duration / dur_sum
            block_durs = [d * scale for d in block_durs]

        current_time = seg_start

        for block_text, block_dur in zip(blocks, block_durs):
            block_end = current_time + block_dur

            subtitle_blocks.append({
                "start": current_time,
                "end": block_end,
                "original_start": current_time,
                "original_end": block_end,
                "text": block_text,
                "lines": split_text_into_lines(
                    block_text,
                    max_line_len=max_line_len,
                ),
                "timing_adjusted": False,
            })

            current_time = block_end

    if adjust_reading_time:
        subtitle_blocks = adjust_timings_for_reading_speed(
            subtitle_blocks,
            max_reading_speed=MAX_READING_SPEED,
            min_gap=MIN_GAP_BETWEEN_SUBS,
        )

    return subtitle_blocks

In [15]:
en_baseline_results = []

for item in whisper_results:
    print(f"Сегментация baseline: {item['folder']}")

    start_time = time.time()

    blocks = segments_to_blocks(
        item["segments"],
        max_chars_block=MAX_CHARS_BLOCK,
        max_line_len=MAX_LINE_LEN,
        max_lines=MAX_LINES,
        min_block_dur=MIN_BLOCK_DUR,
        adjust_reading_time=True,
    )

    elapsed_time = time.time() - start_time

    en_baseline_results.append({
        "folder": item["folder"],
        "variant": "EN_baseline",
        "blocks": blocks,
        "segmentation_time_sec": round(elapsed_time, 4),
    })

    adjusted_count = sum(1 for b in blocks if b.get("timing_adjusted"))

    print("Количество субтитров:", len(blocks))
    print("Сдвинуто окончаний по скорости чтения:", adjusted_count)
    print("Время сегментации:", round(elapsed_time, 4), "сек")

Сегментация baseline: video1
Количество субтитров: 273
Сдвинуто окончаний по скорости чтения: 0
Время сегментации: 0.0563 сек
Сегментация baseline: video10
Количество субтитров: 29
Сдвинуто окончаний по скорости чтения: 0
Время сегментации: 0.0039 сек
Сегментация baseline: video11
Количество субтитров: 77
Сдвинуто окончаний по скорости чтения: 0
Время сегментации: 0.0195 сек
Сегментация baseline: video12
Количество субтитров: 10
Сдвинуто окончаний по скорости чтения: 2
Время сегментации: 0.0011 сек
Сегментация baseline: video13
Количество субтитров: 22
Сдвинуто окончаний по скорости чтения: 1
Время сегментации: 0.0037 сек
Сегментация baseline: video14
Количество субтитров: 15
Сдвинуто окончаний по скорости чтения: 2
Время сегментации: 0.0018 сек
Сегментация baseline: video15
Количество субтитров: 19
Сдвинуто окончаний по скорости чтения: 4
Время сегментации: 0.0019 сек
Сегментация baseline: video2
Количество субтитров: 123
Сдвинуто окончаний по скорости чтения: 0
Время сегментации: 0.0

Метрики для базовых субтитров

In [17]:
import numpy as np
import pandas as pd


def count_limit_violations(blocks):
    line_length_violations = 0
    line_count_violations = 0
    reading_speed_violations = 0
    adjusted_timings_count = 0

    for b in blocks:
        text = b["text"]
        duration = b["end"] - b["start"]

        if b.get("timing_adjusted", False):
            adjusted_timings_count += 1

        if duration > 0:
            cps = len(text) / duration
            if cps > MAX_READING_SPEED:
                reading_speed_violations += 1

        lines = b["lines"]

        if len(lines) > MAX_LINES:
            line_count_violations += 1

        for line in lines:
            if len(line) > MAX_LINE_LEN:
                line_length_violations += 1

    return {
        "line_length_violations": line_length_violations,
        "line_count_violations": line_count_violations,
        "reading_speed_violations": reading_speed_violations,
        "adjusted_timings_count": adjusted_timings_count,
    }


def compute_basic_metrics(results):
    rows = []

    for item in results:
        blocks = item["blocks"]

        lengths = [len(b["text"]) for b in blocks]

        speeds = []
        durations = []

        for b in blocks:
            duration = b["end"] - b["start"]
            if duration > 0:
                durations.append(duration)
                speeds.append(len(b["text"]) / duration)

        violations = count_limit_violations(blocks)

        rows.append({
            "video": item["folder"],
            "variant": item["variant"],
            "blocks_count": len(blocks),
            "avg_subtitle_chars": round(np.mean(lengths), 2) if lengths else 0,
            "avg_duration_sec": round(np.mean(durations), 2) if durations else 0,
            "avg_reading_speed_cps": round(np.mean(speeds), 2) if speeds else 0,
            "max_reading_speed_cps": round(np.max(speeds), 2) if speeds else 0,
            **violations,
            "total_limit_violations": (
                violations["line_length_violations"]
                + violations["line_count_violations"]
                + violations["reading_speed_violations"]
            ),
            "segmentation_time_sec": item["segmentation_time_sec"],
        })

    return pd.DataFrame(rows)


df_baseline_metrics = compute_basic_metrics(en_baseline_results)
df_baseline_metrics

,video,variant,blocks_count,avg_subtitle_chars,avg_duration_sec,avg_reading_speed_cps,max_reading_speed_cps,line_length_violations,line_count_violations,reading_speed_violations,adjusted_timings_count,total_limit_violations,segmentation_time_sec
0,video1,EN_baseline,273,39.66,3.12,12.24,31.68,0,0,59,0,59,0.0563
1,video10,EN_baseline,29,26.66,3.69,7.61,25.09,0,0,2,0,2,0.0039
2,video11,EN_baseline,77,54.69,3.71,14.14,19.71,0,0,14,0,14,0.0195
3,video12,EN_baseline,10,24.60,1.86,11.11,19.20,0,0,3,2,3,0.0011
4,video13,EN_baseline,22,37.91,3.25,11.28,24.66,0,0,3,1,3,0.0037
5,video14,EN_baseline,15,31.93,1.89,17.87,30.00,0,0,8,2,8,0.0018
6,video15,EN_baseline,19,26.68,2.49,11.09,23.50,0,0,3,4,3,0.0019
7,video2,EN_baseline,123,34.17,2.88,13.26,33.05,0,0,30,0,30,0.0165
8,video3,EN_baseline,379,37.66,3.01,13.31,52.90,0,0,104,0,104,0.0613
9,video4,EN_baseline,250,48.88,3.03,15.70,30.16,0,0,105,10,105,0.0450


Семантическая и синтаксическая сегментация

In [27]:
import spacy
from sentence_transformers import SentenceTransformer

model_emb = SentenceTransformer("all-MiniLM-L6-v2")
nlp = spacy.load("en_core_web_sm")

print("Semantic и syntax модели загружены")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Semantic и syntax модели загружены


Семантическая сегментация субтитров

In [28]:
def cosine_sim(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def refresh_block(block):
    new_block = block.copy()
    new_block["text"] = clean_text_for_subs(new_block["text"])
    new_block["lines"] = split_text_into_lines(
        new_block["text"],
        max_line_len=MAX_LINE_LEN,
    )
    return new_block


def block_is_valid(block):
    text = block["text"]
    lines = split_text_into_lines(text, max_line_len=MAX_LINE_LEN)

    return (
        len(text) <= MAX_CHARS_BLOCK
        and len(lines) <= MAX_LINES
        and all(len(line) <= MAX_LINE_LEN for line in lines)
    )


def reading_speed(block):
    duration = block["end"] - block["start"]
    if duration <= 0:
        return float("inf")
    return len(block["text"]) / duration


def semantic_merge_blocks(blocks, threshold=0.82):

    if len(blocks) <= 1:
        return blocks

    blocks = [refresh_block(b) for b in blocks]
    texts = [b["text"] for b in blocks]

    embeddings = model_emb.encode(
        texts,
        batch_size=64,
        show_progress_bar=False,
    )

    merged = []
    i = 0

    while i < len(blocks):
        current = blocks[i]

        if i < len(blocks) - 1:
            next_block = blocks[i + 1]
            sim = cosine_sim(embeddings[i], embeddings[i + 1])

            candidate = {
                "start": current["start"],
                "end": next_block["end"],
                "original_start": current.get("original_start", current["start"]),
                "original_end": next_block.get("original_end", next_block["end"]),
                "text": current["text"] + " " + next_block["text"],
                "timing_adjusted": (
                    current.get("timing_adjusted", False)
                    or next_block.get("timing_adjusted", False)
                ),
            }

            candidate = refresh_block(candidate)

            if (
                sim >= threshold
                and block_is_valid(candidate)
                and reading_speed(candidate) <= MAX_READING_SPEED
            ):
                merged.append(candidate)
                i += 2
                continue

        merged.append(current)
        i += 1

    return merged


def semantic_split_position(text):


    words = text.split()

    if len(words) < 6:
        return None

    candidates = []

    for i in range(2, len(words) - 2):
        left = " ".join(words[:i])
        right = " ".join(words[i:])

        if not is_valid_block_text(left):
            continue

        if not is_valid_block_text(right):
            continue

        emb = model_emb.encode(
            [left, right],
            show_progress_bar=False,
        )

        sim = cosine_sim(emb[0], emb[1])

        balance_penalty = abs(len(left) - len(right)) / max(len(text), 1)

        score = sim + balance_penalty

        candidates.append((score, i))

    if not candidates:
        return None

    candidates.sort(key=lambda x: x[0])
    return candidates[0][1]

Синтаксическая сегментация субтитров

In [40]:
def is_bad_boundary(left_text, right_text):
    left_doc = nlp(left_text)
    right_doc = nlp(right_text)

    if len(left_doc) == 0 or len(right_doc) == 0:
        return False

    left_last = left_doc[-1]
    right_first = right_doc[0]

    left_word = left_last.text.lower()
    right_pos = right_first.pos_

    if left_last.pos_ == "ADP" and right_pos in {"NOUN", "PROPN", "PRON", "DET", "ADJ"}:
        return True

    if left_last.pos_ == "DET" and right_pos in {"NOUN", "PROPN", "ADJ"}:
        return True

    if left_word == "to" and right_pos == "VERB":
        return True

    if left_word in {"not", "n't"} and right_pos in {"VERB", "AUX", "ADJ"}:
        return True

    if left_last.pos_ == "AUX" and right_pos in {"VERB", "AUX"}:
        return True

    if left_last.pos_ == "ADJ" and right_pos in {"NOUN", "PROPN"}:
        return True

    if left_last.pos_ == "PROPN" and right_pos == "PROPN":
        return True

    if left_last.pos_ == "PRON" and right_pos in {"VERB", "AUX"}:
        return True

    return False


def syntax_boundary_refinement(blocks):

    if len(blocks) <= 1:
        return blocks

    refined = []
    i = 0

    while i < len(blocks):
        current = refresh_block(blocks[i])

        if i < len(blocks) - 1:
            next_block = refresh_block(blocks[i + 1])

            if is_bad_boundary(current["text"], next_block["text"]):
                candidate = {
                    "start": current["start"],
                    "end": next_block["end"],
                    "original_start": current.get("original_start", current["start"]),
                    "original_end": next_block.get("original_end", next_block["end"]),
                    "text": current["text"] + " " + next_block["text"],
                    "timing_adjusted": (
                        current.get("timing_adjusted", False)
                        or next_block.get("timing_adjusted", False)
                    ),
                }

                candidate = refresh_block(candidate)

                if (
                    block_is_valid(candidate)
                    and reading_speed(candidate) <= MAX_READING_SPEED
                ):
                    refined.append(candidate)
                    i += 2
                    continue

        refined.append(current)
        i += 1

    return refined

Функции для метрик

In [41]:
def count_bad_syntax_splits(blocks):
    count = 0

    for i in range(len(blocks) - 1):
        left_text = blocks[i]["text"]
        right_text = blocks[i + 1]["text"]

        if is_bad_boundary(left_text, right_text):
            count += 1

    return count


def compute_semantic_coherence(blocks):
    if len(blocks) <= 1:
        return None

    texts = [b["text"] for b in blocks]

    embeddings = model_emb.encode(
        texts,
        batch_size=64,
        show_progress_bar=False,
    )

    similarities = []

    for i in range(len(embeddings) - 1):
        sim = cosine_sim(embeddings[i], embeddings[i + 1])
        similarities.append(sim)

    return round(float(np.mean(similarities)), 4) if similarities else None


def compute_full_metrics(results):
    rows = []

    for item in results:
        blocks = item["blocks"]

        lengths = [len(b["text"]) for b in blocks]

        speeds = []
        durations = []

        for b in blocks:
            duration = b["end"] - b["start"]

            if duration > 0:
                durations.append(duration)
                speeds.append(len(b["text"]) / duration)

        violations = count_limit_violations(blocks)

        bad_syntax_splits = count_bad_syntax_splits(blocks)
        semantic_coherence = compute_semantic_coherence(blocks)

        rows.append({
            "video": item["folder"],
            "variant": item["variant"],
            "blocks_count": len(blocks),
            "avg_subtitle_chars": round(np.mean(lengths), 2) if lengths else 0,
            "avg_duration_sec": round(np.mean(durations), 2) if durations else 0,
            "avg_reading_speed_cps": round(np.mean(speeds), 2) if speeds else 0,
            "max_reading_speed_cps": round(np.max(speeds), 2) if speeds else 0,
            **violations,
            "total_limit_violations": (
                violations["line_length_violations"]
                + violations["line_count_violations"]
                + violations["reading_speed_violations"]
            ),
            "bad_syntax_splits_count": bad_syntax_splits,
            "avg_semantic_coherence": semantic_coherence,
            "segmentation_time_sec": item["segmentation_time_sec"],
        })

    return pd.DataFrame(rows)

Запуск синтаксической сегментации

In [42]:
en_syntax_results = []

for item in en_baseline_results:
    print(f"Syntax-boundary: {item['folder']}")

    start_time = time.time()

    blocks = syntax_boundary_refinement(item["blocks"])

    blocks = adjust_timings_for_reading_speed(
        blocks,
        max_reading_speed=MAX_READING_SPEED,
        min_gap=MIN_GAP_BETWEEN_SUBS,
    )

    elapsed = time.time() - start_time

    en_syntax_results.append({
        "folder": item["folder"],
        "variant": "EN_syntax_boundary",
        "blocks": blocks,
        "segmentation_time_sec": round(elapsed, 4),
    })

    print("Количество субтитров:", len(blocks))
    print("Время:", round(elapsed, 4), "сек")

Syntax-boundary: video1
Количество субтитров: 264
Время: 3.6272 сек
Syntax-boundary: video10
Количество субтитров: 29
Время: 0.3704 сек
Syntax-boundary: video11
Количество субтитров: 77
Время: 1.5926 сек
Syntax-boundary: video12
Количество субтитров: 10
Время: 0.1516 сек
Syntax-boundary: video13
Количество субтитров: 22
Время: 0.424 сек
Syntax-boundary: video14
Количество субтитров: 15
Время: 0.3005 сек
Syntax-boundary: video15
Количество субтитров: 18
Время: 0.3006 сек
Syntax-boundary: video2
Количество субтитров: 123
Время: 2.1813 сек
Syntax-boundary: video3
Количество субтитров: 376
Время: 4.9799 сек
Syntax-boundary: video4
Количество субтитров: 249
Время: 3.3823 сек
Syntax-boundary: video5
Количество субтитров: 376
Время: 6.5971 сек
Syntax-boundary: video6
Количество субтитров: 33
Время: 0.413 сек
Syntax-boundary: video7
Количество субтитров: 408
Время: 5.2767 сек
Syntax-boundary: video8
Количество субтитров: 78
Время: 0.9941 сек
Syntax-boundary: video9
Количество субтитров: 831
Вр

Метрики синтаксической сегментации

In [43]:
df_syntax_metrics = compute_full_metrics(en_syntax_results)
df_syntax_metrics

,video,variant,blocks_count,avg_subtitle_chars,avg_duration_sec,avg_reading_speed_cps,max_reading_speed_cps,line_length_violations,line_count_violations,reading_speed_violations,adjusted_timings_count,total_limit_violations,bad_syntax_splits_count,avg_semantic_coherence,segmentation_time_sec
0,video1,EN_syntax_boundary,264,41.05,3.23,12.39,31.68,0,0,59,0,59,25,0.2478,3.6272
1,video10,EN_syntax_boundary,29,26.66,3.69,7.61,25.09,0,0,2,0,2,0,0.2106,0.3704
2,video11,EN_syntax_boundary,77,54.69,3.71,14.14,19.71,0,0,14,0,14,14,0.2004,1.5926
3,video12,EN_syntax_boundary,10,24.60,1.86,11.11,19.20,0,0,3,1,3,0,0.1834,0.1516
4,video13,EN_syntax_boundary,22,37.91,3.25,11.28,24.66,0,0,3,1,3,2,0.1647,0.4240
5,video14,EN_syntax_boundary,15,31.93,1.89,17.87,30.00,0,0,8,1,8,0,0.1872,0.3005
6,video15,EN_syntax_boundary,18,28.22,2.62,11.45,23.50,0,0,3,2,3,0,0.2776,0.3006
7,video2,EN_syntax_boundary,123,34.17,2.88,13.26,33.05,0,0,30,0,30,8,0.2647,2.1813
8,video3,EN_syntax_boundary,376,37.97,3.04,13.35,52.90,0,0,103,0,103,21,0.2442,4.9799
9,video4,EN_syntax_boundary,249,49.08,3.05,15.71,30.16,0,0,104,3,104,16,0.2465,3.3823


Запуск семантической сегментации

In [44]:
en_semantic_results = []

for item in en_baseline_results:
    print(f"Semantic-only: {item['folder']}")

    start_time = time.time()

    blocks = semantic_merge_blocks(item["blocks"])

    blocks = adjust_timings_for_reading_speed(
        blocks,
        max_reading_speed=MAX_READING_SPEED,
        min_gap=MIN_GAP_BETWEEN_SUBS,
    )

    elapsed = time.time() - start_time

    en_semantic_results.append({
        "folder": item["folder"],
        "variant": "EN_semantic_only",
        "blocks": blocks,
        "segmentation_time_sec": round(elapsed, 4),
    })

    print("Количество субтитров:", len(blocks))
    print("Время:", round(elapsed, 4), "сек")

Semantic-only: video1
Количество субтитров: 271
Время: 2.5054 сек
Semantic-only: video10
Количество субтитров: 28
Время: 0.2369 сек
Semantic-only: video11
Количество субтитров: 77
Время: 0.6587 сек
Semantic-only: video12
Количество субтитров: 10
Время: 0.118 сек
Semantic-only: video13
Количество субтитров: 22
Время: 0.1861 сек
Semantic-only: video14
Количество субтитров: 15
Время: 0.1393 сек
Semantic-only: video15
Количество субтитров: 19
Время: 0.1295 сек
Semantic-only: video2
Количество субтитров: 121
Время: 1.0018 сек
Semantic-only: video3
Количество субтитров: 378
Время: 2.1706 сек
Semantic-only: video4
Количество субтитров: 250
Время: 1.6697 сек
Semantic-only: video5
Количество субтитров: 378
Время: 2.3296 сек
Semantic-only: video6
Количество субтитров: 32
Время: 0.2154 сек
Semantic-only: video7
Количество субтитров: 427
Время: 3.4668 сек
Semantic-only: video8
Количество субтитров: 82
Время: 0.9675 сек
Semantic-only: video9
Количество субтитров: 830
Время: 5.2888 сек


Метрики семантической сегментации

In [45]:
df_semantic_metrics = compute_full_metrics(en_semantic_results)
df_semantic_metrics

,video,variant,blocks_count,avg_subtitle_chars,avg_duration_sec,avg_reading_speed_cps,max_reading_speed_cps,line_length_violations,line_count_violations,reading_speed_violations,adjusted_timings_count,total_limit_violations,bad_syntax_splits_count,avg_semantic_coherence,segmentation_time_sec
0,video1,EN_semantic_only,271,39.96,3.15,12.27,31.68,0,0,59,0,59,34,0.2392,2.5054
1,video10,EN_semantic_only,28,27.64,4.04,7.73,25.09,0,0,2,0,2,0,0.1821,0.2369
2,video11,EN_semantic_only,77,54.69,3.71,14.14,19.71,0,0,14,0,14,14,0.2004,0.6587
3,video12,EN_semantic_only,10,24.60,1.86,11.11,19.20,0,0,3,1,3,0,0.1834,0.1180
4,video13,EN_semantic_only,22,37.91,3.25,11.28,24.66,0,0,3,1,3,2,0.1647,0.1861
5,video14,EN_semantic_only,15,31.93,1.89,17.87,30.00,0,0,8,1,8,0,0.1872,0.1393
6,video15,EN_semantic_only,19,26.68,2.49,11.09,23.50,0,0,3,2,3,1,0.2578,0.1295
7,video2,EN_semantic_only,121,34.75,2.92,13.36,33.05,0,0,30,0,30,8,0.2510,1.0018
8,video3,EN_semantic_only,378,37.76,3.02,13.32,52.90,0,0,104,0,104,24,0.2401,2.1706
9,video4,EN_semantic_only,250,48.88,3.03,15.70,30.16,0,0,105,3,105,17,0.2453,1.6697


Запуск комбинированной сегментации (базовая+семантическая+синтаксическая)

In [46]:
en_syntax_semantic_results = []

for item in en_baseline_results:
    print(f"Syntax+Semantic: {item['folder']}")

    start_time = time.time()

    blocks = syntax_boundary_refinement(item["blocks"])
    blocks = semantic_merge_blocks(blocks)

    blocks = adjust_timings_for_reading_speed(
        blocks,
        max_reading_speed=MAX_READING_SPEED,
        min_gap=MIN_GAP_BETWEEN_SUBS,
    )

    elapsed = time.time() - start_time

    en_syntax_semantic_results.append({
        "folder": item["folder"],
        "variant": "EN_syntax_semantic",
        "blocks": blocks,
        "segmentation_time_sec": round(elapsed, 4),
    })

    print("Количество субтитров:", len(blocks))
    print("Время:", round(elapsed, 4), "сек")

Syntax+Semantic: video1
Количество субтитров: 262
Время: 5.0691 сек
Syntax+Semantic: video10
Количество субтитров: 28
Время: 0.6496 сек
Syntax+Semantic: video11
Количество субтитров: 77
Время: 1.8214 сек
Syntax+Semantic: video12
Количество субтитров: 10
Время: 0.2736 сек
Syntax+Semantic: video13
Количество субтитров: 22
Время: 0.7913 сек
Syntax+Semantic: video14
Количество субтитров: 15
Время: 0.5222 сек
Syntax+Semantic: video15
Количество субтитров: 18
Время: 0.5352 сек
Syntax+Semantic: video2
Количество субтитров: 121
Время: 3.4746 сек
Syntax+Semantic: video3
Количество субтитров: 375
Время: 7.098 сек
Syntax+Semantic: video4
Количество субтитров: 249
Время: 6.6066 сек
Syntax+Semantic: video5
Количество субтитров: 375
Время: 7.3123 сек
Syntax+Semantic: video6
Количество субтитров: 32
Время: 0.6507 сек
Syntax+Semantic: video7
Количество субтитров: 408
Время: 9.452 сек
Syntax+Semantic: video8
Количество субтитров: 78
Время: 1.5771 сек
Syntax+Semantic: video9
Количество субтитров: 829
Вр

Метрики комбинированной сегментации

In [47]:
df_syntax_semantic_metrics = compute_full_metrics(en_syntax_semantic_results)
df_syntax_semantic_metrics

,video,variant,blocks_count,avg_subtitle_chars,avg_duration_sec,avg_reading_speed_cps,max_reading_speed_cps,line_length_violations,line_count_violations,reading_speed_violations,adjusted_timings_count,total_limit_violations,bad_syntax_splits_count,avg_semantic_coherence,segmentation_time_sec
0,video1,EN_syntax_semantic,262,41.37,3.25,12.42,31.68,0,0,59,0,59,25,0.2424,5.0691
1,video10,EN_syntax_semantic,28,27.64,4.04,7.73,25.09,0,0,2,0,2,0,0.1821,0.6496
2,video11,EN_syntax_semantic,77,54.69,3.71,14.14,19.71,0,0,14,0,14,14,0.2004,1.8214
3,video12,EN_syntax_semantic,10,24.60,1.86,11.11,19.20,0,0,3,1,3,0,0.1834,0.2736
4,video13,EN_syntax_semantic,22,37.91,3.25,11.28,24.66,0,0,3,1,3,2,0.1647,0.7913
5,video14,EN_syntax_semantic,15,31.93,1.89,17.87,30.00,0,0,8,1,8,0,0.1872,0.5222
6,video15,EN_syntax_semantic,18,28.22,2.62,11.45,23.50,0,0,3,2,3,0,0.2776,0.5352
7,video2,EN_syntax_semantic,121,34.75,2.92,13.36,33.05,0,0,30,0,30,8,0.2510,3.4746
8,video3,EN_syntax_semantic,375,38.07,3.04,13.37,52.90,0,0,103,0,103,21,0.2420,7.0980
9,video4,EN_syntax_semantic,249,49.08,3.05,15.71,30.16,0,0,104,3,104,16,0.2465,6.6066


Общая таблица всех вариантов

In [49]:
all_segmentation_results_final = (
    en_baseline_results
    + en_syntax_results
    + en_semantic_results
    + en_syntax_semantic_results
)

df_all_segmentation_metrics_final = compute_full_metrics(
    all_segmentation_results_final
)

df_all_segmentation_metrics_final

,video,variant,blocks_count,avg_subtitle_chars,avg_duration_sec,avg_reading_speed_cps,max_reading_speed_cps,line_length_violations,line_count_violations,reading_speed_violations,adjusted_timings_count,total_limit_violations,bad_syntax_splits_count,avg_semantic_coherence,segmentation_time_sec
0,video1,EN_baseline,273,39.66,3.12,12.24,31.68,0,0,59,0,59,34,0.2444,0.0563
1,video10,EN_baseline,29,26.66,3.69,7.61,25.09,0,0,2,0,2,0,0.2106,0.0039
2,video11,EN_baseline,77,54.69,3.71,14.14,19.71,0,0,14,0,14,14,0.2004,0.0195
3,video12,EN_baseline,10,24.60,1.86,11.11,19.20,0,0,3,2,3,0,0.1834,0.0011
4,video13,EN_baseline,22,37.91,3.25,11.28,24.66,0,0,3,1,3,2,0.1647,0.0037
5,video14,EN_baseline,15,31.93,1.89,17.87,30.00,0,0,8,2,8,0,0.1872,0.0018
6,video15,EN_baseline,19,26.68,2.49,11.09,23.50,0,0,3,4,3,1,0.2578,0.0019
7,video2,EN_baseline,123,34.17,2.88,13.26,33.05,0,0,30,0,30,8,0.2647,0.0165
8,video3,EN_baseline,379,37.66,3.01,13.31,52.90,0,0,104,0,104,24,0.2422,0.0613
9,video4,EN_baseline,250,48.88,3.03,15.70,30.16,0,0,105,10,105,17,0.2453,0.0450


Сводные значения всех вариантов

In [50]:
summary_by_variant = (
    df_all_segmentation_metrics_final
    .groupby("variant")
    .agg({
        "blocks_count": "mean",
        "avg_subtitle_chars": "mean",
        "avg_duration_sec": "mean",
        "avg_reading_speed_cps": "mean",
        "max_reading_speed_cps": "mean",
        "line_length_violations": "sum",
        "line_count_violations": "sum",
        "reading_speed_violations": "sum",
        "total_limit_violations": "sum",
        "bad_syntax_splits_count": "sum",
        "avg_semantic_coherence": "mean",
        "segmentation_time_sec": "mean",
    })
    .round(4)
    .reset_index()
)

summary_by_variant

,variant,blocks_count,avg_subtitle_chars,avg_duration_sec,avg_reading_speed_cps,max_reading_speed_cps,line_length_violations,line_count_violations,reading_speed_violations,total_limit_violations,bad_syntax_splits_count,avg_semantic_coherence,segmentation_time_sec
0,EN_baseline,196.6667,37.2747,2.9987,13.0420,30.4813,0,0,1118,1118,218,0.2291,0.0426
1,EN_semantic_only,196.0000,37.4967,3.0393,13.0653,30.4813,0,0,1116,1116,218,0.2241,1.4056
2,EN_syntax_boundary,193.9333,37.8113,3.0387,13.1100,30.4813,0,0,1113,1113,177,0.2322,2.8512
3,EN_syntax_semantic,193.2667,38.0347,3.0787,13.1347,30.4813,0,0,1111,1111,177,0.2273,4.2591


Сравнение времени на обработку

In [51]:
summary_time = (
    df_all_segmentation_metrics_final
    .groupby("variant")
    .agg({
        "segmentation_time_sec": ["mean", "sum"]
    })
)

summary_time.columns = ["avg_time_per_video_sec", "total_time_sec"]

summary_time.reset_index()

,variant,avg_time_per_video_sec,total_time_sec
0,EN_baseline,0.042593,0.6389
1,EN_semantic_only,1.405607,21.0841
2,EN_syntax_boundary,2.851227,42.7684
3,EN_syntax_semantic,4.259073,63.8861
